In [67]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

In [68]:
CUSTOMER_CSV = "customer_dataset.csv"
TRANSACTION_CSV = "transaction_dataset.csv"
OUTPUT_DIR = "./w5/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Thresholds for transaction amounts
LOW_THRESHOLD = 2500.0
HIGH_THRESHOLD = 7500.0

# Common key mappings for city name standardization
CITY_MAP_STANDARD = {
    'Cochin': 'Kochi',
    'Cmbt': 'Coimbatore',
    'Hyd': 'Hyderabad',
    'Trivandrm': 'Trivandrum',
    'Poona': 'Pune'
}

In [69]:
def standardize_city(name):
    if pd.isnull(name):
        return np.nan
    s = str(name).strip().title()
    return CITY_MAP_STANDARD.get(s, s.title())

In [70]:
def classify_amount(amount):
    try:
        a = float(amount)
    except Exception:
        return np.nan
    if a <= 0:
        return np.nan
    if a < LOW_THRESHOLD:
        return "Low"
    elif a <= HIGH_THRESHOLD:
        return "Medium"
    else:
        return "High"

# 1. Load

In [71]:
print("Loading datasets...")
df_customers = pd.read_csv(CUSTOMER_CSV, dtype=str)
df_transactions = pd.read_csv(TRANSACTION_CSV, dtype=str)

print("\nCustomer schema & sample:")
print(df_customers.dtypes)
print(df_customers.head(5))

print("\nTransaction schema & sample:")
print(df_transactions.dtypes)
print(df_transactions.head(5))

cust = df_customers.copy()
txn = df_transactions.copy()

Loading datasets...

Customer schema & sample:
customer_id      object
customer_name    object
city             object
status           object
dtype: object
  customer_id customer_name        city  status
0        c001  Customer_001  TRIVANDRUM  active
1        c002  Customer_002     Chennai  active
2        c003  Customer_003  Trivandrum  active
3        c004  Customer_004  Trivandrum  active
4        c005  Customer_005   Bengaluru  active

Transaction schema & sample:
transaction_id        object
customer_id           object
transaction_date      object
transaction_amount    object
payment_mode          object
dtype: object
  transaction_id customer_id transaction_date transaction_amount payment_mode
0          t0001        c104       2025-01-22               2291       wallet
1          t0002        c137       2025-01-04               9897       wallet
2          t0003        c007       2025-01-11               7222         card
3          t0004        c119       2025-01-24         

In [72]:
pd.to_numeric(txn['transaction_amount']).describe()

count     306.000000
mean     4934.666667
std      2807.435747
min      -300.000000
25%      2554.250000
50%      4971.500000
75%      7266.750000
max      9983.000000
Name: transaction_amount, dtype: float64

# 2. Transform

In [73]:
cust.columns = [c.strip() for c in cust.columns]
txn.columns = [c.strip() for c in txn.columns]

if 'city' in cust.columns:
    cust['city_original'] = cust['city']
    cust['city'] = cust['city'].apply(standardize_city)

if 'transaction_amount' in txn.columns:
    txn['transaction_amount_original'] = txn['transaction_amount']
    txn['transaction_amount'] = pd.to_numeric(txn['transaction_amount'], errors='coerce')

txn['amount_category'] = txn['transaction_amount'].apply(classify_amount)

txn['transaction_date_parsed'] = pd.to_datetime(txn['transaction_date'], errors='coerce')
txn['transaction_month'] = txn['transaction_date_parsed'].dt.month

for df in (cust, txn):
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)


In [74]:
txn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   transaction_id               305 non-null    object        
 1   customer_id                  305 non-null    object        
 2   transaction_date             306 non-null    object        
 3   transaction_amount           306 non-null    int64         
 4   payment_mode                 306 non-null    object        
 5   transaction_amount_original  306 non-null    object        
 6   amount_category              305 non-null    object        
 7   transaction_date_parsed      305 non-null    datetime64[ns]
 8   transaction_month            305 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 21.6+ KB


# 3. Cleaning

In [75]:
customer_invalid_rows = pd.DataFrame(columns=cust.columns)
transaction_invalid_rows = pd.DataFrame(columns=txn.columns)

# (A) Handle duplicates: If records are identical across all columns -> Mark duplicates -> keep first, others invalid
def extract_duplicate_invalids(df, dataset_name):
    dup_mask = df.duplicated(keep='first')
    invalids = df[dup_mask].copy()
    cleaned = df[~dup_mask].copy()
    print(f"{dataset_name}: found {dup_mask.sum()} duplicate rows (kept first instance).")
    return cleaned, invalids

cust_clean, cust_dup_invalids = extract_duplicate_invalids(cust, "Customers")
txn_clean, txn_dup_invalids = extract_duplicate_invalids(txn, "Transactions")
customer_invalid_rows = pd.concat([customer_invalid_rows, cust_dup_invalids], ignore_index=True, sort=False)
transaction_invalid_rows = pd.concat([transaction_invalid_rows, txn_dup_invalids], ignore_index=True, sort=False)

# (B) Invalid customer mask
cust_invalid_mask = (
    cust_clean['customer_id'].isnull() | (cust_clean['customer_id'].astype(str).str.strip() == '') |
    (cust_clean.get('city', pd.Series()).isnull() | (cust_clean['city'].astype(str).str.strip() == '')) |
    (cust_clean.get('status', pd.Series()).isnull() | (cust_clean['status'].astype(str).str.strip() == ''))
)

# (C) Invalid transaction mask
txn_invalid_mask = (
    txn_clean['transaction_id'].isnull() | (txn_clean['transaction_id'].astype(str).str.strip() == '') |
    txn_clean['customer_id'].isnull() | (txn_clean['customer_id'].astype(str).str.strip() == '') |
    txn_clean['transaction_amount'].isnull() | (txn_clean['transaction_amount'] <= 0) |
    txn_clean['transaction_date_parsed'].isnull()
)

# (D) Referential integrity mask
existing_customer_ids = set(cust_clean['customer_id'].astype(str).unique())
txn_clean['customer_id_str'] = txn_clean['customer_id'].astype(str)
ref_invalid_mask = ~txn_clean['customer_id_str'].isin(existing_customer_ids)

# (E) Apply all masks and extract invalid rows
# Customer invalid
cust_invalids = cust_clean[cust_invalid_mask].copy()
cust_clean = cust_clean[~cust_invalid_mask].copy()
print(f"Customers: extracted {len(cust_invalids)} invalid rows (missing key fields).")

# Transaction invalid
txn_invalids = txn_clean[txn_invalid_mask].copy()
txn_clean = txn_clean[~txn_invalid_mask].copy()
print(f"Transactions: extracted {len(txn_invalids)} invalid rows (missing keys, non-positive amounts, or bad dates).")

# Referential integrity invalid transactions
ref_invalids = txn_clean[ref_invalid_mask].copy()
txn_clean = txn_clean[~ref_invalid_mask].copy()
print(f"Transactions: {len(ref_invalids)} rows with customer_id not found in customers (referential integrity).")

# (F) Concatenate invalid rows
customer_invalid_rows = pd.concat([customer_invalid_rows, cust_invalids], ignore_index=True, sort=False)
transaction_invalid_rows = pd.concat([transaction_invalid_rows, txn_invalids, ref_invalids], ignore_index=True, sort=False)

# (G) Verify and format dates as YYYY-MM-DD in cleaned txn
txn_clean['transaction_date'] = txn_clean['transaction_date_parsed'].dt.strftime('%Y-%m-%d')

# Reset indexes on all outputs
cust_clean = cust_clean.reset_index(drop=True)
txn_clean = txn_clean.reset_index(drop=True)
customer_invalid_rows = customer_invalid_rows.reset_index(drop=True)
transaction_invalid_rows = transaction_invalid_rows.reset_index(drop=True)


Customers: found 2 duplicate rows (kept first instance).
Transactions: found 2 duplicate rows (kept first instance).
Customers: extracted 3 invalid rows (missing key fields).
Transactions: extracted 4 invalid rows (missing keys, non-positive amounts, or bad dates).
Transactions: 0 rows with customer_id not found in customers (referential integrity).


C:\Users\shriv\AppData\Local\Temp\ipykernel_4028\1268641750.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  transaction_invalid_rows = pd.concat([transaction_invalid_rows, txn_dup_invalids], ignore_index=True, sort=False)
C:\Users\shriv\AppData\Local\Temp\ipykernel_4028\1268641750.py:49: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ref_invalids = txn_clean[ref_invalid_mask].copy()
C:\Users\shriv\AppData\Local\Temp\ipykernel_4028\1268641750.py:50: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  txn_clean = txn_clean[~ref_invalid_mask].copy()


In [76]:
print("\n--- Sample invalid customer rows ---")
print(customer_invalid_rows.head(5))
print("\n--- Sample cleaned customer rows ---")
print(cust.head(5))

print("\n--- Sample invalid transaction rows ---")
print(transaction_invalid_rows.head(5))
print("\n--- Sample cleaned transaction rows ---")
print(txn.head(5))


--- Sample invalid customer rows ---
  customer_id customer_name        city  status city_original
0        c001  Customer_001  Trivandrum  active    TRIVANDRUM
1        c002  Customer_002     Chennai  active       Chennai
2         NaN  Customer_X01       Kochi  active         Kochi
3        c999  Customer_X02         NaN  active           NaN
4        c998  Customer_X03  Trivandrum     NaN    Trivandrum

--- Sample cleaned customer rows ---
  customer_id customer_name        city  status city_original
0        c001  Customer_001  Trivandrum  active    TRIVANDRUM
1        c002  Customer_002     Chennai  active       Chennai
2        c003  Customer_003  Trivandrum  active    Trivandrum
3        c004  Customer_004  Trivandrum  active    Trivandrum
4        c005  Customer_005   Bengaluru  active     Bengaluru

--- Sample invalid transaction rows ---
  transaction_id customer_id transaction_date transaction_amount payment_mode  \
0          t0001        c104       2025-01-22             

# 4. Join

In [77]:
# Join customers and transactions on customer_id
joined = txn_clean.merge(cust_clean, how='inner', on='customer_id', suffixes=('_txn', '_cust'))

print(f"\nJoined rows: {len(joined)}")
print(joined.head(5))


Joined rows: 300
  transaction_id customer_id transaction_date  transaction_amount  \
0          t0001        c104       2025-01-22                2291   
1          t0002        c137       2025-01-04                9897   
2          t0003        c007       2025-01-11                7222   
3          t0004        c119       2025-01-24                4462   
4          t0005        c084       2025-01-28                5555   

  payment_mode transaction_amount_original amount_category  \
0       wallet                        2291             Low   
1       wallet                        9897            High   
2         card                        7222          Medium   
3   netbanking                        4462          Medium   
4          upi                        5555          Medium   

  transaction_date_parsed  transaction_month customer_id_str customer_name  \
0              2025-01-22                1.0            c104  Customer_104   
1              2025-01-04             

# 5. Aggregations

In [78]:
# Total and average transaction amount per customer
agg_customer = (
    joined.groupby(['customer_id', 'customer_name'], as_index=False)
    .agg(
        total_transaction_amount = ('transaction_amount', 'sum'),
        average_transaction_amount = ('transaction_amount', 'mean'),
        transaction_count = ('transaction_amount', 'count')
    )
    .sort_values('total_transaction_amount', ascending=False)
)

# Total transaction amount per city
if 'city' in joined.columns:
    agg_city = (
        joined.groupby('city', as_index=False)
        .agg(total_transaction_amount = ('transaction_amount', 'sum'),
             transaction_count = ('transaction_amount', 'count'))
        .sort_values('total_transaction_amount', ascending=False)
    )
else:
    agg_city = pd.DataFrame()

# Top 3 customers by total transaction amount
top_3_customers = agg_customer.head(3).copy()

print("\n--- Aggregation: total & avg per customer (sample) ---")
print(agg_customer.head(5))

print("\n--- Aggregation: total per city (sample) ---")
print(agg_city.head(5))

print("\n--- Top 3 customers by total transaction amount ---")
print(top_3_customers)


--- Aggregation: total & avg per customer (sample) ---
   customer_id customer_name  total_transaction_amount  \
88        c111  Customer_111                     40066   
55        c070  Customer_070                     31927   
53        c068  Customer_068                     29998   
26        c033  Customer_033                     29303   
74        c093  Customer_093                     28491   

    average_transaction_amount  transaction_count  
88                 6677.666667                  6  
55                 7981.750000                  4  
53                 5999.600000                  5  
26                 7325.750000                  4  
74                 5698.200000                  5  

--- Aggregation: total per city (sample) ---
         city  total_transaction_amount  transaction_count
3  Coimbatore                    241741                 45
8  Trivandrum                    232665                 45
2     Chennai                    230836                 42
4

In [79]:
cust.to_csv(os.path.join(OUTPUT_DIR, "cleaned_customers.csv"), index=False)
txn.to_csv(os.path.join(OUTPUT_DIR, "cleaned_transactions.csv"), index=False)

# invalid_rows:
customer_invalid_rows.to_csv(os.path.join(OUTPUT_DIR, "invalid_customers.csv"), index=False)
transaction_invalid_rows.to_csv(os.path.join(OUTPUT_DIR, "invalid_transactions.csv"), index=False)

# joined_data
joined.to_csv(os.path.join(OUTPUT_DIR, "joined_data.csv"), index=False)

# aggregates
agg_customer.to_csv(os.path.join(OUTPUT_DIR, "aggregate_total_avg_per_customer.csv"), index=False)
agg_city.to_csv(os.path.join(OUTPUT_DIR, "aggregate_total_per_city.csv"), index=False)
top_3_customers.to_csv(os.path.join(OUTPUT_DIR, "top_3_customers.csv"), index=False)

print(f"\nAll output CSVs written to {OUTPUT_DIR}")
print("Files created:")
for f in os.listdir(OUTPUT_DIR):
    print(" -", f)


All output CSVs written to ./w5/
Files created:
 - aggregate_total_avg_per_customer.csv
 - aggregate_total_per_city.csv
 - cleaned_customers.csv
 - cleaned_transactions.csv
 - invalid_customers.csv
 - invalid_transactions.csv
 - joined_data.csv
 - top_3_customers.csv
